In [3]:
import os
import sys
import time
import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

# Path setup (assume this notebook runs under tagging/pure_unsmear)
THIS_DIR = os.getcwd()
MODULE_DIR = THIS_DIR
REPO_DIR = os.path.abspath(os.path.join(MODULE_DIR, '..'))

sys.path.insert(0, MODULE_DIR)

import tool  # noqa: E402
import importlib
from model import ParticleTransformerKD  # noqa: E402
importlib.reload(tool)  # noqa: E402

# Random seed
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Output directories for this experiment
RUN_NAME = 'single_feature_smear_scan'
OUT_DIR = os.path.join(MODULE_DIR, 'runs', RUN_NAME)
FIG_DIR = os.path.join(OUT_DIR, 'figs')
CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')

tool.ensure_dir(FIG_DIR)
tool.ensure_dir(CKPT_DIR)

CONFIG = {
    'data_path': os.path.join(REPO_DIR, 'test.h5'),
    'n_jets': 200000,
    'max_particles': 100,
    'hlt_effects': {
        'pt_resolution': 0.10,
        'eta_resolution': 0.03,
        'phi_resolution': 0.03,
        'energy_hidden_pt_resolution': 0.10,
        'energy_hidden_eta_resolution': 0.03,
    },
    'tagger': {
        'input_dim': 7,
        'embed_dim': 128,
        'num_heads': 8,
        'num_layers': 4,
        'ff_dim': 512,
        'dropout': 0.1,
    },
    'training': {
        'batch_size': 512,
        'epochs': 50,
        'lr': 5e-4,
        'weight_decay': 1e-5,
        'warmup_epochs': 3,
        'patience': 8,
    },
    'io': {
        'run_name': RUN_NAME,
        'out_dir': OUT_DIR,
        'fig_dir': FIG_DIR,
        'ckpt_dir': CKPT_DIR,
        'config_path': os.path.join(OUT_DIR, 'config.json'),
        'load_baseline': None,
    },
}

tool.save_config(CONFIG, CONFIG['io']['config_path'])
print('Data path:', CONFIG['data_path'])
print('Run dir:', CONFIG['io']['out_dir'])

# Scan configuration
SCAN = {
    'repeats': 3,
    'run_seeds': [100, 101, 102],
    'pt_smear_scales': [0.5, 1.0, 1.5],
    'eta_smear_scales': [0.5, 1.0, 1.5],
    'phi_smear_scales': [0.5, 1.0, 1.5],
    'energy_smear_scales': [0.5, 1.0, 1.5],
    'fpr_grid_n': 201,
}


Device: cuda
Data path: d:\PracticeTagging\tagging\test.h5
Run dir: d:\PracticeTagging\tagging\pure_unsmear\runs\single_feature_smear_scan


In [4]:
# Load data
n = int(CONFIG['n_jets'])
S = int(CONFIG['max_particles'])

with h5py.File(CONFIG['data_path'], 'r') as f:
    labels = f['labels'][:n].astype(np.int64)
    weights = f['weights'][:n].astype(np.float32)
    pt = f['fjet_clus_pt'][:n, :S].astype(np.float32)
    eta = f['fjet_clus_eta'][:n, :S].astype(np.float32)
    phi = f['fjet_clus_phi'][:n, :S].astype(np.float32)
    E = f['fjet_clus_E'][:n, :S].astype(np.float32)

constituents_raw = np.stack([pt, eta, phi, E], axis=-1)  # [N,S,4]
masks_raw = pt > 0

print('Raw:', constituents_raw.shape, 'mask:', masks_raw.shape)
print('Signal:', int(labels.sum()), 'Bkg:', int((1 - labels).sum()))


Raw: (200000, 100, 4) mask: (200000, 100)
Signal: 99836 Bkg: 100164


In [5]:
# Baseline: offline / HLT both use only the original valid tokens (pt > 0)
masks_off = masks_raw.copy()
constituents_off = constituents_raw.copy()
constituents_off[~masks_off] = 0.0

# Split the dataset once and reuse it across all experiments for a fair comparison
idx = np.arange(len(labels))
train_idx, temp_idx = train_test_split(idx, test_size=0.3, random_state=seed, stratify=labels)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=seed, stratify=labels[temp_idx])
print(f"Split: train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,}")


# Implement 7D features locally in this notebook to avoid depending on older external interfaces

def compute_features_local(const: np.ndarray, mask: np.ndarray) -> np.ndarray:
    pt = np.maximum(const[:, :, 0], 1e-8)
    eta = np.clip(const[:, :, 1], -5.0, 5.0)
    phi = const[:, :, 2]
    E = np.maximum(const[:, :, 3], 1e-8)

    px = pt * np.cos(phi)
    py = pt * np.sin(phi)
    pz = pt * np.sinh(eta)

    m = mask.astype(np.float32)
    jet_px = (px * m).sum(axis=1, keepdims=True)
    jet_py = (py * m).sum(axis=1, keepdims=True)
    jet_pz = (pz * m).sum(axis=1, keepdims=True)
    jet_E = (E * m).sum(axis=1, keepdims=True)

    jet_pt = np.sqrt(jet_px**2 + jet_py**2) + 1e-8
    jet_p = np.sqrt(jet_px**2 + jet_py**2 + jet_pz**2) + 1e-8
    jet_eta = 0.5 * np.log(np.clip((jet_p + jet_pz) / (jet_p - jet_pz + 1e-8), 1e-8, 1e8))
    jet_phi = np.arctan2(jet_py, jet_px)

    dEta = eta - jet_eta
    dPhi = np.arctan2(np.sin(phi - jet_phi), np.cos(phi - jet_phi))
    feats = np.stack(
        [
            dEta,
            dPhi,
            np.log(pt + 1e-8),
            np.log(E + 1e-8),
            np.log(pt / jet_pt + 1e-8),
            np.log(E / (jet_E + 1e-8) + 1e-8),
            np.sqrt(dEta**2 + dPhi**2),
        ],
        axis=-1,
    )
    feats = np.clip(np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0), -20.0, 20.0)
    feats[~mask] = 0.0
    return feats.astype(np.float32)


# Single-variable smear generator:
# - pt / eta / phi: change only that quantity and keep E fixed to the offline value
# - energy: internally smear a hidden pt/eta pair, recompute E with a massless approximation,
#   but keep the visible pt/eta/phi values unchanged for the model

def apply_single_smear_variant(
    base_const: np.ndarray,
    base_mask: np.ndarray,
    *,
    smear_target: str = 'none',
    res_scale: float = 1.0,
    run_seed: int = 42,
):
    rs = np.random.RandomState(int(run_seed))
    hlt = np.asarray(base_const, dtype=np.float32).copy()
    hlt_mask = np.asarray(base_mask, dtype=bool).copy()
    valid = hlt_mask.copy()

    base_pt = np.asarray(base_const[:, :, 0], dtype=np.float32)
    base_eta = np.asarray(base_const[:, :, 1], dtype=np.float32)
    base_phi = np.asarray(base_const[:, :, 2], dtype=np.float32)
    base_E = np.asarray(base_const[:, :, 3], dtype=np.float32)

    hlt[:, :, 0] = np.where(valid, base_pt, 0.0)
    hlt[:, :, 1] = np.where(valid, np.clip(base_eta, -5.0, 5.0), 0.0)
    hlt[:, :, 2] = np.where(valid, np.arctan2(np.sin(base_phi), np.cos(base_phi)), 0.0)
    hlt[:, :, 3] = np.where(valid, base_E, 0.0)

    target = str(smear_target).lower()
    if target == 'none':
        pass
    elif target == 'pt':
        pt_noise = rs.normal(
            1.0,
            float(CONFIG['hlt_effects']['pt_resolution']) * float(res_scale),
            size=base_pt.shape,
        )
        pt_noise = np.clip(pt_noise, 0.5, 1.5)
        hlt[:, :, 0] = np.where(valid, base_pt * pt_noise, 0.0)
    elif target == 'eta':
        eta_noise = rs.normal(
            0.0,
            float(CONFIG['hlt_effects']['eta_resolution']) * float(res_scale),
            size=base_eta.shape,
        )
        hlt[:, :, 1] = np.where(valid, np.clip(base_eta + eta_noise, -5.0, 5.0), 0.0)
    elif target == 'phi':
        phi_noise = rs.normal(
            0.0,
            float(CONFIG['hlt_effects']['phi_resolution']) * float(res_scale),
            size=base_phi.shape,
        )
        new_phi = base_phi + phi_noise
        hlt[:, :, 2] = np.where(valid, np.arctan2(np.sin(new_phi), np.cos(new_phi)), 0.0)
    elif target == 'energy':
        pt_noise = rs.normal(
            1.0,
            float(CONFIG['hlt_effects']['energy_hidden_pt_resolution']) * float(res_scale),
            size=base_pt.shape,
        )
        pt_noise = np.clip(pt_noise, 0.5, 1.5)
        eta_noise = rs.normal(
            0.0,
            float(CONFIG['hlt_effects']['energy_hidden_eta_resolution']) * float(res_scale),
            size=base_eta.shape,
        )
        pt_hidden = np.where(valid, base_pt * pt_noise, 0.0)
        eta_hidden = np.where(valid, np.clip(base_eta + eta_noise, -5.0, 5.0), 0.0)
        E_hidden = pt_hidden * np.cosh(np.clip(eta_hidden, -5.0, 5.0))
        hlt[:, :, 3] = np.where(valid, E_hidden, 0.0)
    else:
        raise ValueError(f"Unknown smear_target: {smear_target}")

    hlt = np.nan_to_num(hlt, nan=0.0, posinf=0.0, neginf=0.0)
    hlt[~hlt_mask] = 0.0
    return hlt.astype(np.float32), hlt_mask.astype(bool)


# Precompute offline features (the normalization statistics also come from offline train)
features_off = compute_features_local(constituents_off, masks_off)
print('Offline features:', features_off.shape)


Split: train=140,000 val=30,000 test=30,000
Offline features: (200000, 100, 7)


In [6]:
# Standardize using OFFLINE TRAIN statistics (for comparability)
feat_means_off, feat_stds_off = tool.get_stats(features_off, masks_off, train_idx)

features_off_std = tool.standardize(features_off, masks_off, feat_means_off, feat_stds_off, clip=10.0)

print('Standardization done.')
print('Offline std:', features_off_std.shape)

# Helper: standardize HLT features using the same offline stats (for every scan setting)

def standardize_hlt(features_hlt: np.ndarray, mask_hlt: np.ndarray):
    return tool.standardize(features_hlt, mask_hlt, feat_means_off, feat_stds_off, clip=10.0)

Standardization done.
Offline std: (200000, 100, 7)


In [7]:
# Scan utilities
from pathlib import Path

BS = int(CONFIG['training']['batch_size'])

epochs = int(CONFIG['training']['epochs'])
lr = float(CONFIG['training']['lr'])
wd = float(CONFIG['training']['weight_decay'])
warm = int(CONFIG['training']['warmup_epochs'])
pat = int(CONFIG['training']['patience'])

# Output directories
RES_DIR = tool.ensure_dir(Path(OUT_DIR) / 'results')
CKPT_DIR = Path(CONFIG['io']['ckpt_dir'])


def _fmt_val(v: float) -> str:
    # For filenames (avoid '.')
    s = f"{float(v):.6g}"
    return s.replace('.', 'p').replace('-', 'm')


def _set_seeds(s: int):
    # Training reproducibility (not fully deterministic, but stable enough for variance estimates)
    torch.manual_seed(int(s))
    np.random.seed(int(s))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(s))


def _make_loaders(feat_hlt_std: np.ndarray, mask_hlt: np.ndarray, *, run_seed: int):
    # Make shuffle order reproducible
    g = torch.Generator()
    g.manual_seed(int(run_seed))

    train_ds = tool.JetDataset(
        features_off_std[train_idx],
        feat_hlt_std[train_idx],
        labels[train_idx],
        masks_off[train_idx],
        mask_hlt[train_idx],
        weights[train_idx],
    )
    val_ds = tool.JetDataset(
        features_off_std[val_idx],
        feat_hlt_std[val_idx],
        labels[val_idx],
        masks_off[val_idx],
        mask_hlt[val_idx],
        weights[val_idx],
    )
    test_ds = tool.JetDataset(
        features_off_std[test_idx],
        feat_hlt_std[test_idx],
        labels[test_idx],
        masks_off[test_idx],
        mask_hlt[test_idx],
        weights[test_idx],
    )

    train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, drop_last=True, generator=g)
    val_loader = DataLoader(val_ds, batch_size=BS)
    test_loader = DataLoader(test_ds, batch_size=BS)
    return train_loader, val_loader, test_loader


def _train_or_load(
    name: str,
    train_loader,
    val_loader,
    test_loader,
    *,
    feat_key: str,
    mask_key: str,
    ckpt_path: Path,
    run_seed: int,
):
    _set_seeds(int(run_seed))
    model = ParticleTransformerKD(**CONFIG['tagger']).to(device)

    if ckpt_path.exists():
        tool.load_checkpoint(model, ckpt_path.as_posix(), map_location=device)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sch = tool.get_scheduler(opt, warm, epochs)
        best_auc, best_state, no_imp = 0.0, None, 0
        t0 = time.time()
        for ep in range(1, epochs + 1):
            _loss, _ = tool.train_standard(model, train_loader, opt, device, feat_key, mask_key)
            val_auc, _, _ = tool.evaluate(model, val_loader, device, feat_key, mask_key)
            sch.step()
            if val_auc > best_auc + 1e-4:
                best_auc = float(val_auc)
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_imp = 0
            else:
                no_imp += 1
            if ep == 1 or ep % 5 == 0:
                dt = time.time() - t0
                print(f"[{name}] ep={ep:03d} val_auc={val_auc:.4f} best={best_auc:.4f} no_imp={no_imp} time={dt:.1f}s")
            if no_imp >= pat:
                print(f"[{name}] Early stopping")
                break
        if best_state is not None:
            model.load_state_dict(best_state)
        tool.save_checkpoint(model, ckpt_path.as_posix(), extra={'best_val_auc': float(best_auc), 'run_seed': int(run_seed)})
        print('Saved checkpoint:', ckpt_path.as_posix())

    auc, p, y = tool.evaluate(model, test_loader, device, feat_key, mask_key)
    return float(auc), np.asarray(p), np.asarray(y)


def _run_model_setting(
    *,
    tag: str,
    feat_hlt_std: np.ndarray,
    mask_hlt: np.ndarray,
    run_seed: int,
    feat_key: str,
    mask_key: str,
):
    # Cache: one result file per setting + seed
    res_path = Path(RES_DIR) / f"roc_{tag}_seed{int(run_seed)}.npz"
    if res_path.exists():
        data = np.load(res_path)
        return data['fpr'], data['tpr'], float(data['auc'])

    ckpt_path = CKPT_DIR / f"tagger_{tag}_seed{int(run_seed)}.pt"

    train_loader, val_loader, test_loader = _make_loaders(feat_hlt_std, mask_hlt, run_seed=int(run_seed))
    auc, p, y = _train_or_load(
        tag,
        train_loader,
        val_loader,
        test_loader,
        feat_key=feat_key,
        mask_key=mask_key,
        ckpt_path=ckpt_path,
        run_seed=int(run_seed),
    )

    fpr, tpr, auc2 = tool.compute_roc(y, p)
    np.savez(res_path, fpr=fpr, tpr=tpr, auc=float(auc2))
    print('Saved ROC:', res_path.as_posix())
    return fpr, tpr, float(auc2)


def run_offline_teacher_once():
    # Train or load the offline teacher only once; all smear plots share this baseline.
    results = {}
    for s in SCAN['run_seeds']:
        fpr, tpr, auc = _run_model_setting(
            tag='offline_teacher',
            feat_hlt_std=features_off_std,
            mask_hlt=masks_off,
            run_seed=int(s),
            feat_key='off',
            mask_key='mask_off',
        )
        results[int(s)] = (fpr, tpr, float(auc))
    return results


def _interp_tpr(fpr: np.ndarray, tpr: np.ndarray, grid: np.ndarray) -> np.ndarray:
    # Interpolate to a shared FPR grid (for mean/std across repeats)
    # Requires fpr to be monotonically increasing
    return np.interp(grid, fpr, tpr)


In [ ]:
# Run the single-variable smear scan (repeat each setting 3 times)

TARGET_SCAN_VALUES = {
    'pt': SCAN['pt_smear_scales'],
    'eta': SCAN['eta_smear_scales'],
    'phi': SCAN['phi_smear_scales'],
    'energy': SCAN['energy_smear_scales'],
}

TARGET_LABELS = {
    'pt': 'pT only',
    'eta': 'eta only',
    'phi': 'phi only',
    'energy': 'E-only-randomized',
}


def _make_hlt_std_for_setting(*, setting: dict, run_seed: int):
    const_hlt, mask_hlt = apply_single_smear_variant(
        constituents_off,
        masks_off,
        smear_target=str(setting.get('smear_target', 'none')),
        res_scale=float(setting.get('res_scale', 1.0)),
        run_seed=int(run_seed),
    )
    feat = compute_features_local(const_hlt, mask_hlt)
    feat_std = standardize_hlt(feat, mask_hlt)
    return feat_std, mask_hlt


def run_single_smear_scan(smear_target: str, values):
    results = {}
    for v in values:
        results[float(v)] = {}
        for s in SCAN['run_seeds']:
            setting = {
                'smear_target': str(smear_target),
                'res_scale': float(v),
            }
            feat_std, mask_hlt = _make_hlt_std_for_setting(setting=setting, run_seed=int(s))
            tag = f"{smear_target}_{_fmt_val(float(v))}"
            fpr, tpr, auc = _run_model_setting(
                tag=tag,
                feat_hlt_std=feat_std,
                mask_hlt=mask_hlt,
                run_seed=int(s),
                feat_key='hlt',
                mask_key='mask_hlt',
            )
            results[float(v)][int(s)] = (fpr, tpr, float(auc))
    return results


print('Running offline teacher once')
offline_teacher_results = run_offline_teacher_once()

print('Running scan: pt only')
res_pt = run_single_smear_scan('pt', SCAN['pt_smear_scales'])

print('Running scan: eta only')
res_eta = run_single_smear_scan('eta', SCAN['eta_smear_scales'])

print('Running scan: phi only')
res_phi = run_single_smear_scan('phi', SCAN['phi_smear_scales'])

print('Running scan: E-only-randomized')
res_energy = run_single_smear_scan('energy', SCAN['energy_smear_scales'])

scan_results = {
    'pt': res_pt,
    'eta': res_eta,
    'phi': res_phi,
    'energy': res_energy,
}


In [ ]:
# Plotting: one ROC plot per physical quantity; each curve is the mean over 3 seeds at a fixed scale
# The black curve is the mean ROC of the offline teacher
# x-axis: TPR (signal efficiency)
# y-axis: FPR (background efficiency, log scale)

tpr_grid = np.linspace(0.0, 1.0, int(SCAN.get('fpr_grid_n', 201)))
EPS = 1e-12


def _interp_fpr_at_tpr(fpr: np.ndarray, tpr: np.ndarray, grid: np.ndarray) -> np.ndarray:
    fpr = np.asarray(fpr)
    tpr = np.asarray(tpr)
    order = np.argsort(tpr)
    t = tpr[order]
    f = fpr[order]
    uniq_t, idx = np.unique(t, return_index=True)
    uniq_f = f[idx]
    if uniq_t.shape[0] < 2:
        return np.full_like(grid, fill_value=float(uniq_f[0]) if uniq_f.size else 1.0, dtype=np.float64)
    return np.interp(grid, uniq_t, uniq_f)


def _mean_curve_fpr(res_by_seed: dict):
    fprs = []
    aucs = []
    for _, (fpr, tpr, auc) in res_by_seed.items():
        fprs.append(_interp_fpr_at_tpr(np.asarray(fpr), np.asarray(tpr), tpr_grid))
        aucs.append(float(auc))
    fprs = np.stack(fprs, axis=0)

    lf = np.log10(np.clip(fprs, EPS, 1.0))
    lf_m = lf.mean(axis=0)
    lf_s = lf.std(axis=0)
    f_m = np.clip(10.0 ** lf_m, EPS, 1.0)
    f_lo = np.clip(10.0 ** (lf_m - lf_s), EPS, 1.0)
    f_hi = np.clip(10.0 ** (lf_m + lf_s), EPS, 1.0)

    aucs = np.asarray(aucs, dtype=np.float64)
    return f_m, f_lo, f_hi, float(aucs.mean()), float(aucs.std())


def plot_effect(effect_name: str, values):
    res = scan_results[effect_name]
    label_name = TARGET_LABELS.get(effect_name, effect_name)

    base_f, base_lo, base_hi, base_auc_m, base_auc_s = _mean_curve_fpr(offline_teacher_results)

    plt.figure(figsize=(7.0, 6.0))
    plt.semilogy(tpr_grid, base_f, color='black', lw=2.6, label=f"Offline teacher (AUC={base_auc_m:.4f}±{base_auc_s:.4f})")
    plt.fill_between(tpr_grid, base_lo, base_hi, color='black', alpha=0.12)

    cmap = plt.get_cmap('viridis')
    vals = list(values)
    for i, v in enumerate(vals):
        f_m, f_lo, f_hi, auc_m, auc_s = _mean_curve_fpr(res[float(v)])
        c = cmap(i / max(1, len(vals) - 1))
        plt.semilogy(tpr_grid, f_m, color=c, lw=1.9, label=f"{label_name} scale={v} (AUC={auc_m:.4f}±{auc_s:.4f})")
        plt.fill_between(tpr_grid, f_lo, f_hi, color=c, alpha=0.10)

    plt.xlabel('True Positive Rate (Signal efficiency)')
    plt.ylabel('False Positive Rate')
    plt.title(f"ROC scan (log FPR): {label_name} (test)")
    plt.ylim(1e-4, 1.0)
    plt.xlim(0.0, 1.0)
    plt.grid(True, which='both', alpha=0.3)
    plt.legend(loc='upper left', fontsize=8)
    plt.tight_layout()

    out_path = os.path.join(CONFIG['io']['fig_dir'], f"roc_scan_{effect_name}_logfpr.png")
    plt.savefig(out_path, dpi=180, bbox_inches='tight')
    print('Saved figure:', out_path)
    plt.show()


def plot_combined_nominal_scale(nominal_scale: float = 1.0):
    # The summary plot compares different physical quantities only at the same chosen scale, without averaging across scales.
    plt.figure(figsize=(7.0, 6.0))

    base_f, base_lo, base_hi, base_auc_m, base_auc_s = _mean_curve_fpr(offline_teacher_results)
    plt.semilogy(tpr_grid, base_f, color='black', lw=2.6, label=f"Offline teacher (AUC={base_auc_m:.4f}±{base_auc_s:.4f})")
    plt.fill_between(tpr_grid, base_lo, base_hi, color='black', alpha=0.12)

    colors = {
        'pt': '#1f77b4',
        'eta': '#ff7f0e',
        'phi': '#2ca02c',
        'energy': '#d62728',
    }
    for effect_name, values in TARGET_SCAN_VALUES.items():
        if len(values) == 0:
            continue
        if float(nominal_scale) in scan_results[effect_name]:
            v = float(nominal_scale)
        else:
            v = float(values[0])
        f_m, f_lo, f_hi, auc_m, auc_s = _mean_curve_fpr(scan_results[effect_name][v])
        label_name = TARGET_LABELS.get(effect_name, effect_name)
        plt.semilogy(
            tpr_grid,
            f_m,
            color=colors.get(effect_name, None),
            lw=1.9,
            label=f"{label_name} scale={v} (AUC={auc_m:.4f}±{auc_s:.4f})",
        )
        plt.fill_between(tpr_grid, f_lo, f_hi, color=colors.get(effect_name, None), alpha=0.10)

    plt.xlabel('True Positive Rate (Signal efficiency)')
    plt.ylabel('False Positive Rate')
    plt.title('ROC comparison: single-feature smear (test)')
    plt.ylim(1e-4, 1.0)
    plt.xlim(0.0, 1.0)
    plt.grid(True, which='both', alpha=0.3)
    plt.legend(loc='upper left', fontsize=8)
    plt.tight_layout()

    out_path = os.path.join(CONFIG['io']['fig_dir'], 'roc_scan_single_feature_comparison_logfpr.png')
    plt.savefig(out_path, dpi=180, bbox_inches='tight')
    print('Saved figure:', out_path)
    plt.show()


for effect_name, values in TARGET_SCAN_VALUES.items():
    plot_effect(effect_name, values)

plot_combined_nominal_scale(1.0)
